# Leafcheck: build the dataset and train (Google Colab)

1. **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom (Runtime > Run all works too).
2. Maize images come from PlantVillage, Kaggle and optionally your own photos. Non-corn images (`Not_Leaf`) come from the other crops in PlantVillage,
   everyday photos (Imagenette), flowers (Oxford Flowers) and optionally your own photos.
3. At the end you download a zip with the model, class names and the unfamiliar-image check. Copy those into `backend/model/`.

If a download cell prints `SKIPPED`, that source was not available. The notebook carries on with the others.
Always check each dataset's licence before you use it for anything beyond coursework.

In [ ]:
# 1) GPU check and packages
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
!pip -q install kagglehub scikit-learn matplotlib tensorflow-datasets

## 2) Training code (the tested scripts, written to this Colab session)

In [ ]:
%%writefile train.py
#!/usr/bin/env python3
"""Train the corn leaf classifier (MobileNetV2 transfer learning), following the project synopsis.

Pipeline
  1. Scan one or more dataset folders, map folder names to classes, drop unreadable images.
  2. Normalise every image to an RGB JPEG (EXIF-rotated, max side 512) in a cache folder.
  3. Remove exact duplicates (MD5) and visual duplicates (dHash); drop images whose duplicates carry different labels.
  4. Stratified train/val/test split (70/15/15) BEFORE any augmentation, saved to splits.csv.
  5. Stage 1: train the new head on a frozen backbone. Stage 2: fine-tune the top backbone layers at a small LR.
  6. Evaluate ONCE on the test split: accuracy, per-class precision/recall/F1, macro-F1, confusion matrix, size, latency.
  7. Export a Keras model + class_names.json that the Leafcheck backend can load with no code changes.

Model contract with the backend (do not change): input 224x224x3 RAW 0-255 pixels; layers named
`rescaling`, `mobilenetv2_1.00_224`, `global_average_pooling2d`, `dropout`, `dense`; classes in alphabetical order.

Usage
  python train.py --data /path/to/dataset [/path/to/more_data ...] --out output
"""
import argparse
import csv
import hashlib
import io
import json
import os
import random
import re
import shutil
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import keras  # noqa: E402
sys.path.insert(0, str(Path(__file__).resolve().parent))  # find fit_ood.py next to this file
import fit_ood  # noqa: E402
import tensorflow as tf  # noqa: E402
from keras import layers  # noqa: E402
from sklearn.metrics import classification_report, confusion_matrix, f1_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402
from sklearn.utils.class_weight import compute_class_weight  # noqa: E402

IMG = 224
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif"}
CACHE_MAX_SIDE = 512


# ---------------------------------------------------------------- classes
def canon(folder: str, overrides: dict[str, str]) -> str | None:
    """Map a dataset folder name to one of our class names (PlantVillage, Kaggle and plain names all work)."""
    if folder in overrides:
        return overrides[folder]
    k = re.sub(r"[^a-z0-9]+", "_", folder.lower()).strip("_")
    tokens = set(k.split("_"))
    if tokens & {"not", "non", "other", "others", "negative", "background"}:
        return "Not_Leaf"
    if "rust" in k:
        return "Common_Rust"
    if "blight" in k:
        return "Blight"
    if tokens & {"gray", "grey", "cercospora"}:
        return "Gray_Leaf_Spot"
    if "healthy" in k:
        return "Healthy"
    return None


def find_images(roots, overrides, skip=()):
    items, unknown = [], Counter()
    for root in roots:
        for d in sorted(Path(root).rglob("*")):
            if not d.is_dir() or d.name in skip:
                continue
            files = [f for f in d.iterdir() if f.suffix.lower() in EXTS]
            if not files:
                continue
            cls = canon(d.name, overrides)
            if cls is None:
                unknown[str(d)] += len(files)
                continue
            items += [(f, cls) for f in files]
    if unknown:
        lines = "\n".join(f"  {p}  ({n} images)" for p, n in unknown.items())
        sys.exit(
            "These folders could not be mapped to a class:\n" + lines +
            "\nRename them, or map them explicitly, e.g.  --map \"Tomato_healthy=Not_Leaf\"  "
            "(or --skip \"<folder name>\" to ignore a folder).")
    return items


# ---------------------------------------------------------------- cleaning
def dhash(img: Image.Image) -> int:
    g = np.asarray(img.convert("L").resize((9, 8), Image.LANCZOS), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


def process(args):
    src, cls, cache = args
    try:
        raw = Path(src).read_bytes()
        img = Image.open(io.BytesIO(raw))
        img.load()
        img = ImageOps.exif_transpose(img).convert("RGB")
        if min(img.size) < 32:
            return None
        md5 = hashlib.md5(raw).hexdigest()
        h = dhash(img)
        img.thumbnail((CACHE_MAX_SIDE, CACHE_MAX_SIDE), Image.BILINEAR)
        out = Path(cache) / cls / f"{md5}.jpg"
        out.parent.mkdir(parents=True, exist_ok=True)
        if not out.exists():
            img.save(out, "JPEG", quality=95)
        return dict(path=str(out), label=cls, md5=md5, dhash=h, src=str(src))
    except Exception:
        return None


def clean(items, cache, workers):
    with ThreadPoolExecutor(workers) as ex:
        rows = list(ex.map(process, [(s, c, cache) for s, c in items]))
    ok = [r for r in rows if r]
    report = {"found": len(items), "unreadable_or_tiny": len(items) - len(ok)}

    def dedupe(rows_in, key):
        """Keep one image per key; if the same image carries different labels it is a label error: drop all."""
        groups = defaultdict(list)
        for r in rows_in:
            groups[r[key]].append(r)
        kept, dupes, conflicts = [], 0, 0
        for g in groups.values():
            if len({r["label"] for r in g}) > 1:
                conflicts += len(g)
                continue
            kept.append(g[0])
            dupes += len(g) - 1
        return kept, dupes, conflicts

    uniq, exact, c1 = dedupe(ok, "md5")  # identical files
    final, visual, c2 = dedupe(uniq, "dhash")  # resized / re-encoded copies
    report["exact_duplicates_removed"] = exact
    report["visual_duplicates_removed"] = visual
    report["label_conflicts_removed"] = c1 + c2
    report["kept"] = len(final)
    return final, report


# ---------------------------------------------------------------- data pipeline
def make_ds(paths, y, batch, training, seed):
    ds = tf.data.Dataset.from_tensor_slices((paths, y))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def load(p, label):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(img, (IMG, IMG))  # squash, same as the backend
        img = tf.cast(img, tf.float32)  # RAW 0-255; the model rescales internally
        img.set_shape((IMG, IMG, 3))
        return img, label

    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch)
    if training:  # augmentation: training split only
        aug = keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.08),
            layers.RandomZoom(0.15),
            layers.RandomContrast(0.2),
            layers.RandomBrightness(0.2, value_range=(0, 255)),
        ])
        ds = ds.map(lambda x, l: (aug(x, training=True), l), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


# ---------------------------------------------------------------- model
def build_model(n_classes, pretrained, dropout):
    inp = keras.Input((IMG, IMG, 3))
    x = layers.Rescaling(1 / 127.5, offset=-1, name="rescaling")(inp)
    base = keras.applications.MobileNetV2(
        input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet" if pretrained else None)
    assert base.name == "mobilenetv2_1.00_224", base.name
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling2d")(x)
    x = layers.Dropout(dropout, name="dropout")(x)
    out = layers.Dense(n_classes, activation="softmax", name="dense")(x)
    return keras.Model(inp, out), base


def fit(model, train, val, lr, epochs, cw, patience):
    model.compile(keras.optimizers.Adam(lr), "sparse_categorical_crossentropy", metrics=["accuracy"])
    cbs = [
        keras.callbacks.EarlyStopping("val_loss", patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=max(1, patience // 2), min_lr=1e-7),
    ]
    h = model.fit(train, validation_data=val, epochs=epochs, class_weight=cw, callbacks=cbs, verbose=2, shuffle=False)
    return {k: [float(v) for v in vs] for k, vs in h.history.items()}


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", nargs="+", required=True, help="dataset root folder(s); sub-folders are classes")
    ap.add_argument("--out", default="output")
    ap.add_argument("--map", action="append", default=[], metavar="FOLDER=CLASS", help="explicit folder->class")
    ap.add_argument("--skip", action="append", default=[], metavar="FOLDER", help="folder name to ignore")
    ap.add_argument("--max-per-class", type=int, default=0, help="cap images per class (0 = no cap); useful for Not_Leaf")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--epochs1", type=int, default=12)
    ap.add_argument("--epochs2", type=int, default=20)
    ap.add_argument("--lr1", type=float, default=1e-3)
    ap.add_argument("--lr2", type=float, default=1e-5)
    ap.add_argument("--unfreeze", type=int, default=40, help="number of top backbone layers to fine-tune")
    ap.add_argument("--dropout", type=float, default=0.25)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--no-pretrained", action="store_true", help="skip ImageNet weights (smoke tests only)")
    ap.add_argument("--mlflow", action="store_true", help="log to MLflow if installed")
    a = ap.parse_args()

    random.seed(a.seed); np.random.seed(a.seed); tf.random.set_seed(a.seed)
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    overrides = dict(m.split("=", 1) for m in a.map)

    # 1-3: scan, clean, dedupe
    items = find_images(a.data, overrides, set(a.skip))
    rows, report = clean(items, out / "clean", a.workers)
    if a.max_per_class:
        rng = random.Random(a.seed); rng.shuffle(rows)
        kept, n = [], Counter()
        for r in rows:
            if n[r["label"]] < a.max_per_class:
                n[r["label"]] += 1; kept.append(r)
        rows = kept
    class_names = sorted({r["label"] for r in rows})
    counts = Counter(r["label"] for r in rows)
    print("cleaning:", json.dumps(report)); print("class counts:", dict(counts))
    too_small = [c for c, n in counts.items() if n < 20]
    if len(class_names) < 2 or too_small:
        sys.exit(f"Need at least 20 images per class and 2+ classes. Too small: {too_small or class_names}")
    if "Not_Leaf" not in class_names:
        print("WARNING: no Not_Leaf class, the model will label any photo as a corn leaf.")
    idx = {c: i for i, c in enumerate(class_names)}
    paths = np.array([r["path"] for r in rows]); y = np.array([idx[r["label"]] for r in rows])

    # 4: stratified split BEFORE augmentation
    tr_p, tmp_p, tr_y, tmp_y = train_test_split(paths, y, test_size=0.30, stratify=y, random_state=a.seed)
    va_p, te_p, va_y, te_y = train_test_split(tmp_p, tmp_y, test_size=0.50, stratify=tmp_y, random_state=a.seed)
    with open(out / "splits.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["path", "label", "split"])
        for name, ps, ys in (("train", tr_p, tr_y), ("val", va_p, va_y), ("test", te_p, te_y)):
            w.writerows([[p, class_names[i], name] for p, i in zip(ps, ys)])
    print(f"split: train={len(tr_p)} val={len(va_p)} test={len(te_p)}")
    train, val, test = (make_ds(tr_p, tr_y, a.batch, True, a.seed), make_ds(va_p, va_y, a.batch, False, a.seed),
                        make_ds(te_p, te_y, a.batch, False, a.seed))
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(len(class_names)), y=tr_y)))

    # 5: two-stage training
    model, base = build_model(len(class_names), not a.no_pretrained, a.dropout)
    base.trainable = False
    print("== Stage 1: head only ==")
    h1 = fit(model, train, val, a.lr1, a.epochs1, cw, patience=4)
    base.trainable = True
    for layer in base.layers[:-a.unfreeze]:
        layer.trainable = False
    for layer in base.layers:  # keep BatchNorm statistics frozen while fine-tuning
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    print(f"== Stage 2: fine-tune top {a.unfreeze} backbone layers ==")
    h2 = fit(model, train, val, a.lr2, a.epochs2, cw, patience=5)

    # 6: evaluate once on the test split
    prob = model.predict(test, verbose=0)
    pred = prob.argmax(1)
    rep = classification_report(te_y, pred, target_names=class_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(te_y, pred)
    sample = np.zeros((1, IMG, IMG, 3), "float32")
    model.predict(sample, verbose=0)
    t0 = time.perf_counter()
    for _ in range(30):
        model(sample, training=False)
    latency_ms = (time.perf_counter() - t0) / 30 * 1000

    # 7: export + verify the backend contract
    model_path = out / "corn_leaf_disease_model_v3.keras"
    model.save(model_path)
    (out / "class_names.json").write_text(json.dumps(class_names, indent=4))
    reloaded = keras.models.load_model(model_path, compile=False)
    for name in ("rescaling", "mobilenetv2_1.00_224", "global_average_pooling2d", "dropout", "dense"):
        reloaded.get_layer(name)
    assert reloaded.get_layer("dense").units == len(class_names)
    batch = next(iter(test.take(1)))[0]
    assert np.allclose(reloaded.predict(batch, verbose=0), model.predict(batch, verbose=0), atol=1e-4)

    # unfamiliar-image check for the backend (corn-class feature statistics; threshold set on the validation split)
    names = lambda yy: [class_names[i] for i in yy]
    ood = fit_ood.fit_from_splits(
        model, {"train": (list(tr_p), names(tr_y)), "val": (list(va_p), names(va_y)), "test": (list(te_p), names(te_y))},
        out / "ood_stats.npz")
    fit_ood.print_report(ood)

    metrics = {
        "ood_check": ood,
        "classes": class_names, "cleaning": report, "class_counts": dict(counts),
        "split_sizes": {"train": len(tr_p), "val": len(va_p), "test": len(te_p)},
        "test_accuracy": float((pred == te_y).mean()),
        "test_macro_f1": float(f1_score(te_y, pred, average="macro")),
        "per_class": {c: rep[c] for c in class_names},
        "confusion_matrix": cm.tolist(),
        "model_size_mb": round(model_path.stat().st_size / 1e6, 2),
        "params": int(model.count_params()),
        "latency_ms_per_image_cpu_or_gpu": round(latency_ms, 1),
        "args": vars(a),
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (out / "history.json").write_text(json.dumps({"stage1": h1, "stage2": h2}))
    try:
        import matplotlib; matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow(cm, cmap="Greens")
        ax.set_xticks(range(len(class_names)), class_names, rotation=30, ha="right"); ax.set_yticks(range(len(class_names)), class_names)
        for i in range(len(cm)):
            for j in range(len(cm)):
                ax.text(j, i, cm[i, j], ha="center", va="center")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); fig.tight_layout(); fig.savefig(out / "confusion_matrix.png", dpi=150)
    except Exception as e:  # plotting is optional
        print("confusion matrix plot skipped:", e)
    if a.mlflow:
        import mlflow
        with mlflow.start_run():
            mlflow.log_params({k: v for k, v in vars(a).items() if k != "data"})
            mlflow.log_metrics({"test_accuracy": metrics["test_accuracy"], "test_macro_f1": metrics["test_macro_f1"]})
            mlflow.log_artifacts(str(out))

    print(f"\nTEST accuracy={metrics['test_accuracy']:.4f} macro-F1={metrics['test_macro_f1']:.4f}")
    print(classification_report(te_y, pred, target_names=class_names, zero_division=0))
    print("confusion matrix (rows=true):\n", cm)
    print(f"\nSaved to {out}/: corn_leaf_disease_model_v3.keras, class_names.json, ood_stats.npz, metrics.json, splits.csv")
    print("Deploy: copy the .keras, class_names.json and ood_stats.npz into backend/model/ and update MODEL_PATH (or replace the old files).")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile fit_ood.py
#!/usr/bin/env python3
"""Fit the "is this a corn leaf?" check that the backend uses (model/ood_stats.npz).

How it works: for every corn-leaf training image, take the model's pooled 1280-d feature vector. Store the class means and a shared
covariance. At prediction time the backend measures how far a new photo's feature vector is from the nearest corn-class mean
(Mahalanobis distance). Photos much farther than almost every real corn leaf are reported as "not a corn leaf".
The threshold is the Nth percentile of distances on HELD-OUT corn leaves, so about (100-N)% of real leaves are rejected by design.

Works with any model that follows the backend contract (including your current v2 model).

Examples
  # current model, from your dataset folders. Folders that map to Not_Leaf (or --negatives) are used only to MEASURE rejection.
  python fit_ood.py --model ../backend/model/corn_leaf_disease_model_v3.keras --data /data/corn /data/not_corn --out ../backend/model/ood_stats.npz

  # right after train.py (re-uses its exact splits)
  python fit_ood.py --model output/corn_leaf_disease_model_v3.keras --splits output/splits.csv --out output/ood_stats.npz
"""
import argparse
import csv
import json
import os
import sys
from pathlib import Path

import numpy as np
from PIL import Image

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import keras  # noqa: E402

IMG = 224
NEG = "Not_Leaf"


def extract(model, paths, batch=64):
    """Pooled 1280-d features, preprocessed exactly like the backend (PIL bilinear resize, raw 0-255)."""
    ext = keras.Model(model.input, model.get_layer("global_average_pooling2d").output)
    out = []
    for i in range(0, len(paths), batch):
        arr = np.stack([np.asarray(Image.open(p).convert("RGB").resize((IMG, IMG), Image.BILINEAR), dtype="float32")
                        for p in paths[i:i + batch]])
        out.append(ext.predict(arr, verbose=0))
    return np.concatenate(out).astype(np.float64)


def distances(f, means, precision):
    best = None
    for mu in means:
        d = f - mu
        v = np.einsum("nd,nd->n", d @ precision, d)
        best = v if best is None else np.minimum(best, v)
    return best


def fit_gate(f_train, y_train, shrinkage=0.1):
    classes = sorted({str(c) for c in y_train})
    y = np.asarray(y_train)
    means = np.stack([f_train[y == c].mean(0) for c in classes])
    centred = f_train - means[[classes.index(c) for c in y_train]]
    cov = centred.T @ centred / max(len(f_train) - 1, 1)
    d = cov.shape[0]
    cov = (1 - shrinkage) * cov + shrinkage * np.trace(cov) / d * np.eye(d)  # shrinkage keeps it invertible and stable
    return classes, means, np.linalg.inv(cov)


def fit_from_splits(model, splits, out_path, percentile=99.0):
    """splits = {"train": (paths, names), "val": (...), "test": (...)}; names are class names, Not_Leaf = negative.

    Fits on corn classes of `train`, sets the threshold on corn images of `val`, and reports held-out rejection rates on `test`
    (plus any Not_Leaf images anywhere, which are never used for fitting).
    """
    def feats(key, positive):
        paths, names = splits[key]
        keep = [i for i, n in enumerate(names) if (n != NEG) == positive]
        return (extract(model, [paths[i] for i in keep]) if keep else np.zeros((0, 1280))), [names[i] for i in keep]

    f_tr, y_tr = feats("train", True)
    f_va, _ = feats("val", True)
    if len(f_tr) < 50 or len(f_va) < 20:
        raise SystemExit("Not enough corn images to fit the check (need >= 50 train and >= 20 validation).")
    classes, means, prec = fit_gate(f_tr, y_tr)
    threshold = float(np.percentile(distances(f_va, means, prec), percentile))

    rep = {"classes": classes, "percentile": percentile, "threshold": threshold, "train_images": len(f_tr)}
    f_te, _ = feats("test", True)
    if len(f_te):
        rep["heldout_corn_rejected"] = float((distances(f_te, means, prec) > threshold).mean())
        rep["heldout_corn_images"] = len(f_te)
    neg = [feats(k, False)[0] for k in ("train", "val", "test")]
    f_neg = np.concatenate([n for n in neg if len(n)]) if any(len(n) for n in neg) else np.zeros((0, 1280))
    if len(f_neg):
        rep["non_corn_rejected"] = float((distances(f_neg, means, prec) > threshold).mean())
        rep["non_corn_images"] = len(f_neg)
    np.savez(out_path, means=means.astype("float32"), precision=prec.astype("float32"),
             threshold=np.float64(threshold), classes=np.array(classes))
    return rep


def print_report(rep):
    print("\n== Unfamiliar-image check ==")
    print(f"fitted on {rep['train_images']} corn images, classes {rep['classes']}, threshold at the {rep['percentile']}th percentile")
    if "heldout_corn_rejected" in rep:
        print(f"real corn leaves wrongly rejected (held-out, n={rep['heldout_corn_images']}): {rep['heldout_corn_rejected']:.1%}")
    if "non_corn_rejected" in rep:
        print(f"non-corn images correctly rejected (n={rep['non_corn_images']}): {rep['non_corn_rejected']:.1%}")
    else:
        print("no non-corn images were available, so the rejection rate for them is NOT measured (pass --negatives or a Not_Leaf folder).")
    print("Tune on the server with OOD_SCALE: raise it (e.g. 1.5) if real leaves get rejected, lower it if non-corn photos get through.")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--model", required=True)
    ap.add_argument("--out", default="ood_stats.npz")
    ap.add_argument("--splits", help="splits.csv written by train.py")
    ap.add_argument("--data", nargs="+", help="dataset root folder(s), same layout as train.py")
    ap.add_argument("--negatives", nargs="*", default=[], help="folders of NON-corn images, only used to measure rejection")
    ap.add_argument("--map", action="append", default=[], metavar="FOLDER=CLASS")
    ap.add_argument("--skip", action="append", default=[])
    ap.add_argument("--percentile", type=float, default=99.0)
    ap.add_argument("--cache", default="ood_cache")
    ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()
    if bool(a.splits) == bool(a.data):
        sys.exit("Give exactly one of --splits or --data")

    model = keras.models.load_model(a.model, compile=False)
    if a.splits:
        sp = {"train": ([], []), "val": ([], []), "test": ([], [])}
        with open(a.splits) as f:
            for r in csv.DictReader(f):
                sp[r["split"]][0].append(r["path"]); sp[r["split"]][1].append(r["label"])
    else:
        from sklearn.model_selection import train_test_split
        sys.path.insert(0, str(Path(__file__).resolve().parent))
        from train import EXTS, clean, find_images  # same scanning and cleaning as training

        items = find_images(a.data, dict(m.split("=", 1) for m in a.map), set(a.skip))
        for d in a.negatives:
            items += [(f, NEG) for f in Path(d).rglob("*") if f.suffix.lower() in EXTS]
        rows, report = clean(items, Path(a.cache), 8)
        print("cleaning:", json.dumps(report))
        pos = [r for r in rows if r["label"] != NEG]
        neg = [r for r in rows if r["label"] == NEG]
        p = np.array([r["path"] for r in pos]); y = np.array([r["label"] for r in pos])
        tr_p, tmp_p, tr_y, tmp_y = train_test_split(p, y, test_size=0.30, stratify=y, random_state=a.seed)
        va_p, te_p, va_y, te_y = train_test_split(tmp_p, tmp_y, test_size=0.50, stratify=tmp_y, random_state=a.seed)
        sp = {"train": (list(tr_p), list(tr_y)), "val": (list(va_p), list(va_y)),
              "test": (list(te_p) + [r["path"] for r in neg], list(te_y) + [NEG] * len(neg))}
        print("NOTE: if this model was already trained on these images, the threshold is optimistic. "
              "Prefer images the model never saw, or use --splits from a fresh train.py run.")
    rep = fit_from_splits(model, sp, a.out, a.percentile)
    print_report(rep)
    print(f"\nSaved {a.out}. Copy it to backend/model/ood_stats.npz and restart the backend.")


if __name__ == "__main__":
    main()

## 3) Settings. Edit this cell if you want different sources or sizes

In [ ]:
from pathlib import Path
import random, shutil, os, json

BASE = Path("/content")
SRC, DATA, OUT = BASE / "src", BASE / "data", BASE / "output"   # downloads / merged dataset / training results
SEED = 42
random.seed(SEED)

# Kaggle datasets with MAIZE leaves (folder names like Blight, Common_Rust, Gray_Leaf_Spot, Healthy).
# Check each name on kaggle.com and add more as "owner/dataset-name".
KAGGLE_CORN = ["smaranjitghose/corn-or-maize-leaf-disease-dataset"]
# Optional Kaggle datasets with NON-corn images (other plants, objects). Every image in them becomes Not_Leaf.
KAGGLE_NON_CORN = []

PV_NON_CORN_PER_FOLDER = 40   # images taken from each non-corn PlantVillage folder (about 34 folders)
IMAGENETTE_N = 400            # everyday photos (objects, animals, vehicles)
FLOWERS_N = 200               # flowers

# Your own photos (optional). Put them in Google Drive and list the folders here.
USE_DRIVE = False
MY_CORN_ROOTS = []       # folders that contain class sub-folders, e.g. ".../corn_field/Blight", ".../corn_field/Healthy"
MY_NON_CORN_ROOTS = []   # folders of non-corn photos (any layout)

In [ ]:
# 4) Helper functions
import re
from PIL import Image

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif"}

def canon(folder):
    """Folder name -> class name (same rules as train.py)."""
    k = re.sub(r"[^a-z0-9]+", "_", folder.lower()).strip("_")
    t = set(k.split("_"))
    if t & {"not", "non", "other", "others", "negative", "background"}: return "Not_Leaf"
    if "rust" in k: return "Common_Rust"
    if "blight" in k: return "Blight"
    if t & {"gray", "grey", "cercospora"}: return "Gray_Leaf_Spot"
    if "healthy" in k: return "Healthy"
    return None

def add_images(root, prefix, force_class=None, cap=None, folder_filter=None):
    """Copy images from root (and its sub-folders) into DATA/<class>/. Returns number copied."""
    root, copied = Path(root), 0
    dirs = [root] + sorted(d for d in root.rglob("*") if d.is_dir())
    for d in dirs:
        if folder_filter and not folder_filter(d.name): continue
        files = [f for f in d.iterdir() if f.is_file() and f.suffix.lower() in EXTS]
        if not files: continue
        cls = force_class or canon(d.name)
        if cls is None:
            print("  skipped unrecognised folder:", d, f"({len(files)} images)"); continue
        random.shuffle(files)
        if cap: files = files[:cap]
        out = DATA / cls; out.mkdir(parents=True, exist_ok=True)
        for f in files:
            shutil.copy2(f, out / f"{prefix}__{re.sub(r'[^A-Za-z0-9]+', '', d.name)[:24]}__{f.name}")
        copied += len(files)
    return copied

def kaggle_get(slug):
    import kagglehub
    try:
        path = kagglehub.dataset_download(slug)
        print("downloaded", slug, "->", path); return Path(path)
    except Exception as e:
        print("SKIPPED", slug, "->", str(e)[:200]); return None

def dump_tfds(name, split, n, class_dir, prefix):
    """Save n random images of a TensorFlow Datasets dataset as JPEGs into DATA/<class_dir>/."""
    import tensorflow_datasets as tfds
    try:
        ds = tfds.load(name, split=split, shuffle_files=True).shuffle(2000, seed=SEED).take(n)
        out = DATA / class_dir; out.mkdir(parents=True, exist_ok=True)
        k = 0
        for ex in tfds.as_numpy(ds):
            Image.fromarray(ex["image"]).convert("RGB").save(out / f"{prefix}__{k}.jpg", quality=92); k += 1
        print(f"saved {k} images from {name}"); return k
    except Exception as e:
        print("SKIPPED", name, "->", str(e)[:200]); return 0

def count_classes():
    return {d.name: len([f for f in d.iterdir() if f.suffix.lower() in EXTS]) for d in sorted(DATA.iterdir()) if d.is_dir()}

## 5) Download the sources

In [ ]:
# 5a) PlantVillage from GitHub (only the colour images folder). It holds corn AND many other crops.
SRC.mkdir(parents=True, exist_ok=True)
pv = SRC / "plantvillage"
if not pv.exists():
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git {pv}
    !cd {pv} && git sparse-checkout set raw/color
PV_COLOR = pv / "raw" / "color"
if PV_COLOR.exists():
    print(len(list(PV_COLOR.iterdir())), "PlantVillage folders, e.g.", sorted(p.name for p in PV_COLOR.iterdir())[:3])
else:
    print("SKIPPED PlantVillage: folder raw/color not found. Contents:", [p.name for p in pv.iterdir()])

In [ ]:
# 5b) MAIZE images -> DATA/Blight, Common_Rust, Gray_Leaf_Spot, Healthy
shutil.rmtree(DATA, ignore_errors=True); DATA.mkdir(parents=True)

if PV_COLOR.exists():
    n = add_images(PV_COLOR, "plantvillage", folder_filter=lambda name: name.lower().startswith("corn"))
    print("PlantVillage maize images:", n)

for slug in KAGGLE_CORN:
    p = kaggle_get(slug)
    if p: print(f"  {slug}: copied", add_images(p, "kaggle_" + slug.split("/")[-1][:12]))

if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    for r in MY_CORN_ROOTS: print("  my maize photos:", r, add_images(r, "mine"))

print(count_classes())

In [ ]:
# 5c) NON-CORN images -> DATA/Not_Leaf
# Other crops from PlantVillage are the most useful: they look like leaves but are not corn.
if PV_COLOR.exists():
    n = add_images(PV_COLOR, "pv_other", force_class="Not_Leaf", cap=PV_NON_CORN_PER_FOLDER,
                   folder_filter=lambda name: not name.lower().startswith("corn"))
    print("other-crop leaves:", n)

dump_tfds("imagenette/320px-v2", "train", IMAGENETTE_N, "Not_Leaf", "imagenette")   # everyday objects, animals, vehicles
dump_tfds("oxford_flowers102", "train", FLOWERS_N, "Not_Leaf", "flowers")            # flowers

for slug in KAGGLE_NON_CORN:
    p = kaggle_get(slug)
    if p: print(f"  {slug}: copied", add_images(p, "kaggle_neg", force_class="Not_Leaf"))

if USE_DRIVE:
    for r in MY_NON_CORN_ROOTS: print("  my non-corn photos:", r, add_images(r, "mine_neg", force_class="Not_Leaf"))

print(count_classes())

## 6) Check the dataset before training
The table below must show all five classes. Very different class sizes are handled by class weights, and `--max-per-class` caps the biggest ones.
PlantVillage photos are taken on plain backgrounds. If you have real field photos of corn, add them through `MY_CORN_ROOTS`, because lab-style data alone usually performs worse in the field.

In [ ]:
counts = count_classes()
for k, v in counts.items(): print(f"{k:16s} {v:6d}")
need = {"Blight", "Common_Rust", "Gray_Leaf_Spot", "Healthy", "Not_Leaf"}
missing = need - set(counts)
assert not missing, f"Missing classes: {missing}. Fix the download cells above before training."
assert min(counts.values()) >= 50, "Some class has fewer than 50 images, so add more data."

## 7) Train (cleans, removes duplicates, splits, trains in two stages, tests once, fits the unfamiliar-image check)

In [ ]:
!python train.py --data {DATA} --out {OUT} --max-per-class 1500 --workers 4

## 8) Results and download

In [ ]:
m = json.load(open(OUT / "metrics.json"))
print("Test accuracy :", round(m["test_accuracy"], 4))
print("Test macro-F1 :", round(m["test_macro_f1"], 4))
print("Class counts  :", m["class_counts"])
print("Cleaning      :", m["cleaning"])
for c, r in m["per_class"].items(): print(f"{c:16s} precision {r['precision']:.3f}  recall {r['recall']:.3f}  f1 {r['f1-score']:.3f}")
o = m.get("ood_check", {})
print("\nReal corn leaves wrongly rejected:", o.get("heldout_corn_rejected"), "| non-corn images rejected:", o.get("non_corn_rejected"))
from IPython.display import Image as show
show(str(OUT / "confusion_matrix.png"))

In [ ]:
# Package only what the backend needs, then download it.
deploy = BASE / "deploy"; shutil.rmtree(deploy, ignore_errors=True); deploy.mkdir()
for f in ["corn_leaf_disease_model_v3.keras", "class_names.json", "ood_stats.npz", "metrics.json", "splits.csv", "confusion_matrix.png"]:
    shutil.copy2(OUT / f, deploy / f)
shutil.make_archive(str(BASE / "leafcheck_model"), "zip", deploy)
from google.colab import files
files.download(str(BASE / "leafcheck_model.zip"))
print("Copy corn_leaf_disease_model_v3.keras, class_names.json and ood_stats.npz into backend/model/ and restart the backend.")